In [ ]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pickle
from rank import rank_qoe, rank_str_to_int_mapping, rank_impact
from charts import plot_confusion_matrix

# Load the dataset
features = ['src2dst_avg_pkt_size', 'src2dst_avg_pkt_size_mean', 'src2dst_avg_pkt_size_median', 'src2dst_avg_pkt_size_std', 'src2dst_avg_pkt_size_min', 'src2dst_avg_pkt_size_max', 'src2dst_pps', 'src2dst_pps_mean', 'src2dst_pps_median', 'src2dst_pps_std', 'src2dst_pps_min', 'src2dst_pps_max', 'dst2src_avg_pkt_size', 'dst2src_avg_pkt_size_mean', 'dst2src_avg_pkt_size_median', 'dst2src_avg_pkt_size_std', 'dst2src_avg_pkt_size_min', 'dst2src_avg_pkt_size_max', 'dst2src_pps', 'dst2src_pps_mean', 'dst2src_pps_median', 'dst2src_pps_std', 'dst2src_pps_min', 'dst2src_pps_max', 'pkt_dir_ratio', 'pkt_dir_ratio_mean', 'pkt_dir_ratio_median', 'pkt_dir_ratio_std', 'pkt_dir_ratio_min', 'pkt_dir_ratio_max']
dataset = pd.read_csv('/Users/<user>/PhD/github/qoe-research/exp/17-May-2024/window.csv') 
dataset.dropna(subset=features, inplace=True)
dataset['y_qoe_by_ping'] = dataset['ping'].apply(lambda ping: rank_str_to_int_mapping[rank_qoe(ping)])



In [ ]:
import matplotlib.pyplot as plt
print("Default DPI:", plt.rcParams['figure.dpi'])
plt.rcParams['figure.dpi'] = 400

In [ ]:
len(dataset)

In [ ]:
exp_df = pd.read_csv('/Users/<user>/PhD/github/qoe-research/exp/17-May-2024/summary.csv')
print(len(exp_df))
# dataset['weapon'] = dataset.merge(exp_df, on='exp_id')['weapon']
# dataset['weapon'].value_counts()

In [ ]:
dataset[dataset['real_player'] == 'adrian']

In [ ]:
print(len(dataset))
X_train = dataset[dataset['exp_id'] % 4 != 0][features]
y_train_by_ping = dataset[dataset['exp_id'] % 4 != 0]['y_qoe_by_ping']
X_test = dataset[dataset['exp_id'] % 4 == 0][features]
y_test_by_ping = dataset[dataset['exp_id'] % 4 == 0]['y_qoe_by_ping']

# X_train = dataset[(dataset['real_player'] != 'adrian') & (dataset['exp_id'] != 67)][features]
# y_train_by_ping = dataset[(dataset['real_player'] != 'adrian') & (dataset['exp_id'] != 67)]['y_qoe_by_ping']
# X_test = dataset[(dataset['real_player'] == 'adrian') | (dataset['exp_id'] == 67)][features]
# y_test_by_ping = dataset[(dataset['real_player'] == 'adrian') | (dataset['exp_id'] == 67)]['y_qoe_by_ping']

# X_train = dataset[dataset['exp_id'] < 59][features]
# y_train_by_ping = dataset[dataset['exp_id'] < 59]['y_qoe_by_ping']
# X_test = dataset[dataset['exp_id'] >= 59][features]
# y_test_by_ping = dataset[dataset['exp_id'] >= 59]['y_qoe_by_ping']

# X = dataset[features]
# y_qoe_by_ping = dataset['ping'].apply(lambda ping: rank_str_to_int_mapping[rank_qoe(ping)])
# X_train, X_test, y_train_by_ping, y_test_by_ping = train_test_split(X, y_qoe_by_ping, test_size=0.4, random_state=42)

y_train_by_survey = dataset.merge(y_train_by_ping, left_index=True, right_index=True, how='right', sort=False)['self_rate_qoe_score']
y_test_by_survey = dataset.merge(y_test_by_ping, left_index=True, right_index=True, how='right', sort=False)['self_rate_qoe_score']
y_train_by_impact = dataset.merge(y_train_by_ping, left_index=True, right_index=True, how='right', sort=False)['impact'].apply(lambda impact: rank_str_to_int_mapping[rank_impact(impact)])
y_test_by_impact = dataset.merge(y_test_by_ping, left_index=True, right_index=True, how='right', sort=False)['impact'].apply(lambda impact: rank_str_to_int_mapping[rank_impact(impact)])
y_train_by_performance = dataset.merge(y_train_by_ping, left_index=True, right_index=True, how='right', sort=False)['self_rate_player_performance']
y_test_by_performance = dataset.merge(y_test_by_ping, left_index=True, right_index=True, how='right', sort=False)['self_rate_player_performance']

In [ ]:
print(len(X_train) + len(X_test),len(X_train), len(X_test))

In [ ]:
def weapon_to_class(weapon):
    if weapon == 'all':
        return 1
    elif weapon == 'only with knife':
        return 2
    return 1

y_train_by_weapon = dataset.merge(y_train_by_ping, left_index=True, right_index=True, how='right', sort=False)['weapon'].apply(lambda weapon: weapon_to_class(weapon))
y_test_by_weapon = dataset.merge(y_test_by_ping, left_index=True, right_index=True, how='right', sort=False)['weapon'].apply(lambda weapon: weapon_to_class(weapon))

In [ ]:
# ping_clf = RandomForestClassifier()
# survey_clf = RandomForestClassifier()
# performance_clf = RandomForestClassifier()
# weapon_clf = RandomForestClassifier()

# Train the models
# ping_clf.fit(X_train, y_train_by_ping)
# survey_clf.fit(X_train, y_train_by_survey)
# performance_clf.fit(X_train, y_train_by_performance)
# weapon_clf.fit(X_train, y_train_by_weapon)

# y_pred_by_ping = ping_clf.predict(X_test)
# y_pred_by_survey = survey_clf.predict(X_test)
# y_pred_by_performance = performance_clf.predict(X_test)
# y_pred_by_weapon = weapon_clf.predict(X_test)

In [ ]:
rank_int_to_str_mapping = {
    5: 'Excellent',
    4: 'Good',
    3: 'Fair',
    2: 'Poor',
    1: 'Bad'
}
# y_test = y_test_by_ping.apply(lambda x: rank_int_to_str_mapping[x])
# y_pred = pd.Series(y_pred_by_survey).apply(lambda x: rank_int_to_str_mapping[x])

In [ ]:
# len(y_test), len(y_test_by_ping)

In [ ]:
# y_test_by_ping.value_counts()

In [ ]:
# y_train_by_ping.value_counts()

In [ ]:
y_train_by_ping.apply(lambda x: 1 if x <= 3 else 2).value_counts()

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

list_of_models = [
    ('RF', RandomForestClassifier()),
    ('DT', DecisionTreeClassifier()),
    ('LR', LogisticRegression()),
    ('SVM', SVC())
]

trained_model = []

result_list = []
target_list = [('ping', y_train_by_ping, y_test_by_ping), ('survey', y_train_by_survey, y_test_by_survey)] #, ('performance', y_train_by_performance, y_test_by_performance)]
for target_name, y_train, y_test_raw in target_list:
    print('Target:', target_name)
    y_train_list = [y_train_by_ping, y_train_by_ping.apply(lambda x: 'a1-3' if x <= 3 else 'a4-5')]
    y_test_list = [y_test_raw, y_test_raw.apply(lambda x: 'a1-3' if x <= 3 else 'a4-5')]
    for i in range(len(y_train_list)):
        y_train = y_train_list[i]
        y_test = y_test_list[i]
        for model_name, model in list_of_models:
            print('Training:', model_name)
            model.fit(X_train, y_train)
        # for model_name, model in list_of_models:
            print('Testing:', model_name)
            y_pred = model.predict(X_test)
            report = classification_report(y_test, y_pred, digits=2, output_dict=True)
            trained_model.append({'target_name': target_name, 'model_name': model_name, 'model': model, 'y_test': y_test, 'y_pred': y_pred, 'report': report})
            for k, v in report.items():
                print(k, v)
                if k not in ('accuracy', 'macro avg', 'weighted avg'):
                    res = {
                        'target': target_name,
                        'model': model_name,
                        'classes': y_test.nunique(),
                        'class': k,
                        'precision': round(v['precision'], 2),
                        'recall': round(v['recall'], 2),
                        'f1-score': round(v['f1-score'], 2)
                    }
                    result_list.append(res)
                    
result_df = pd.DataFrame(result_list)
# Plot the confusion matrix



In [ ]:
pivot_ml_cls = result_df.pivot_table(index=['target', 'class'], columns='model', values=['recall', 'precision', 'f1-score'])
pivot_ml_cls.to_latex('evaluation_table.tex', float_format="%.2f", index_names=False, longtable=True)

In [ ]:
pivot_ml_cls

In [ ]:
trained_model[0]['y_test'].apply(lambda x: rank_int_to_str_mapping[x])

In [ ]:
len(trained_model[0]['y_test']), len(trained_model[0]['y_test'])

In [ ]:
import scikitplot as skplt
labels = ['Bad', 'Poor', 'Fair', 'Good', 'Excellent']
rank_int_to_str_mapping = {
    5: 'Excellent',
    4: 'Good',
    3: 'Fair',
    2: 'Poor',
    1: 'Bad'
}
i = 8
# y_test = y_test_by_ping.apply(lambda x: rank_int_to_str_mapping[x])
# y_pred = pd.Series(y_pred_by_survey).apply(lambda x: rank_int_to_str_mapping[x])
skplt.metrics.plot_confusion_matrix(
    trained_model[i]['y_test'].apply(lambda x: rank_int_to_str_mapping[x]),
    pd.Series(trained_model[i]['y_pred']).apply(lambda x: rank_int_to_str_mapping[x]),
    normalize=True,
    labels=labels
)

trained_model[i]['model_name'], trained_model[i]['target_name']

In [ ]:
import numpy as np
# New confusion matrix extracted from the image
conf_matrix_new = np.array([
    [0.02, 0.90, 0.08, 0.00, 0.00],
    [0.03, 0.80, 0.17, 0.01, 0.00],
    [0.01, 0.37, 0.56, 0.06, 0.00],
    [0.00, 0.00, 0.08, 0.79, 0.13],
    [0.00, 0.01, 0.01, 0.43, 0.55]
])

# Recalculate TP, FP, FN, Precision, Recall, and F1-score
TP_new = np.diag(conf_matrix_new)
FP_new = np.sum(conf_matrix_new, axis=0) - TP_new
FN_new = np.sum(conf_matrix_new, axis=1) - TP_new

precision_new = TP_new / (TP_new + FP_new)
recall_new = TP_new / (TP_new + FN_new)
f1_score_new = 2 * (precision_new * recall_new) / (precision_new + recall_new)

# Store results
precision_recall_f1_new = {
    labels[i]: {
        "Precision": precision_new[i],
        "Recall": recall_new[i],
        "F1-score": f1_score_new[i]
    } for i in range(len(labels))
}

sanity_df = pd.DataFrame(precision_recall_f1_new).T

In [ ]:
for k, v in report.items():
    print(k, v)
    if k not in ('accuracy', 'macro avg', 'weighted avg'):
        res = {
            'model': 'RandomForest',
            'classes': y_test.nunique(),
            'class': k,
            'precision': round(v['precision'], 2),
            'recall': round(v['recall'], 2),
            'f1-score': round(v['f1-score'], 2)
        }
        print(res)
 

In [ ]:
import scikitplot as skplt
labels = ['Bad', 'Poor', 'Fair', 'Good', 'Excellent']

obj = skplt.metrics.plot_confusion_matrix(
    y_test,
    y_pred,
    normalize=False,
    labels=labels,
)

obj.title.set_text('')

In [ ]:
len(y_pred_by_ping), len(y_test_by_ping)

In [ ]:
import scikitplot as skplt
labels = ['Bad', 'Poor', 'Fair', 'Good', 'Excellent']

y_test = y_test_by_ping.apply(lambda x: rank_int_to_str_mapping[x])
y_pred = pd.Series(y_pred_by_ping).apply(lambda x: rank_int_to_str_mapping[x])

obj = skplt.metrics.plot_confusion_matrix(
    y_test,
    y_pred,
    normalize=True,
    labels=labels,
)

obj.title.set_text('')


# obj.title.set_text('Confusion matrix for ping rate prediction')
# obj.title = 'Confusion matrix for QoE by ping'
# Set a custom title
# obj.title('Custom Confusion Matrix Title')


In [ ]:
len(X_train), len(y_test)


In [ ]:
import scikitplot as skplt
labels = ['Bad', 'Poor', 'Fair', 'Good', 'Excellent']

y_test = y_test_by_survey.apply(lambda x: rank_int_to_str_mapping[x])
y_pred = pd.Series(y_pred_by_survey).apply(lambda x: rank_int_to_str_mapping[x])

obj = skplt.metrics.plot_confusion_matrix(
    y_test,
    y_pred,
    normalize=True,
    labels=labels,
)

obj.title.set_text('')


In [ ]:
plot_confusion_matrix(y_test_by_ping, y_pred_by_ping)

In [ ]:
# plot_confusion_matrix(y_test_by_ping, y_pred_by_ping)
# plot_confusion_matrix(y_test_by_survey, y_pred_by_survey)
# plot_confusion_matrix(y_test_by_performance, y_pred_by_performance)
plot_confusion_matrix(y_test_by_weapon, y_pred_by_weapon)